In [3]:
# -*- coding: utf-8 -*-
"""
Make ONE monthly panel that contains:
 - 8 primary emotions mapped from 44 labels in TWO ways at review-level:
     * mean-mapping  → P_{EMO}_mean
     * max-mapping   → P_{EMO}_max
   Then aggregated by movie×month using monthly AVERAGE (same-month reviews averaged)
 - polarity aggregated by movie×month as BOTH:
     * polarity_median (monthly median)
     * polarity_mean   (monthly mean)
 - meta columns, review_count, month_index, gvar_month (from first post_netflix==1 per movie)
 - movie-level months gap from open_date to ott_date: ott_release_time_months (constant per movie)

Output:
  - movie_monthly_panel_emotions_polarity.csv
  - movie_monthly_panel_emotions_polarity.dta
"""

import pandas as pd
import numpy as np

# =========================
# 0) SETTINGS / PATH
# =========================
INPUT_CSV = "movie_long_with_emotions and polarity_full.csv"  # 반드시 polarity_pm1 포함
RELEASE_CSV = "ott_release_by_movie(영화별 출시기간).csv"  # open/ott 날짜, 영화 키
STATA_VERSION = 118
META_COLS = ["OTT공개일", "대표국적", "장르"]

# (선택) 달력 범위 고정 (예: 2020-01~2025-12)
FIXED_RANGE_START = "2020-01-01"  # "2020-01-01"
FIXED_RANGE_END   = "2025-06-01"  # "2025-12-01"

# =========================
# 1) Emotion column mapping
# =========================
ALT = lambda *xs: list(xs)

JOY = ALT("기쁨_prob","행복_prob","즐거움/신남_prob","즐거움_신남_prob",
          "흐뭇함(귀여움/예쁨)_prob","흐뭇함_귀여움_예쁨__prob",
          "뿌듯함_prob","감동/감탄_prob","감동_감탄_prob","편안/쾌적_prob","편안_쾌적_prob")
TRUST = ALT("안심/신뢰_prob","안심_신뢰_prob","환영/호의_prob","환영_호의_prob",
            "존경_prob","아껴주는_prob")
FEAR  = ALT("공포/무서움_prob","공포_무서움_prob","불안/걱정_prob","불안_걱정_prob",
            "부담/안_내킴_prob","부담_안_내킴_prob")
SURPRISE = ALT("놀람_prob","경악_prob","당황/난처_prob","당황_난처_prob","깨달음_prob")
SADNESS  = ALT("슬픔_prob","절망_prob","서러움_prob","안타까움/실망_prob","안타까움_실망_prob",
               "재미없음_prob","힘듦/지침_prob","힘듦_지침_prob","패배/자기혐오_prob",
               "죄책감_prob","불쌍함/연민_prob","불쌍함_연민_prob","부끄러움_prob")
DISGUST  = ALT("역겨움/징그러움_prob","역겨움_징그러움_prob","한심함_prob","어이없음_prob",
               "지긋지긋_prob","증오/혐오_prob","증오_혐오_prob","우쭐댐/무시함_prob","우쭐댐_무시함_prob")
ANGER    = ALT("화남/분노_prob","화남_분노_prob","짜증_prob","불평/불만_prob","불평_불만_prob",
               "증오/혐오_prob","증오_혐오_prob","우쭐댐/무시함_prob","우쭐댐_무시함_prob")
ANTICIP  = ALT("기대감_prob","신기함/관심_prob","신기함_관심_prob","의심/불신_prob","의심_불신_prob")

PRIMARY_EMOTIONS = ["JOY","TRUST","FEAR","SURPRISE","SADNESS","DISGUST","ANGER","ANTICIP"]

GROUPS = {
    "JOY": JOY, "TRUST": TRUST, "FEAR": FEAR, "SURPRISE": SURPRISE,
    "SADNESS": SADNESS, "DISGUST": DISGUST, "ANGER": ANGER, "ANTICIP": ANTICIP
}

# 혼합(가중치 분배: mean-매핑에선 가중합/가중치, max-매핑에선 가중치 곱한 값으로 후보에 포함)
BLEND = {
    "증오/혐오_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "증오_혐오_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "우쭐댐/무시함_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "우쭐댐_무시함_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "고마움_prob": {"JOY": 0.5, "TRUST": 0.5},
    "부끄러움_prob": {"SADNESS": 0.5, "FEAR": 0.5},
}

# =========================
# 2) Utilities
# =========================
def read_csv_kor(path: str) -> pd.DataFrame:
    for enc in ["utf-8-sig","cp949","euc-kr","utf-8","latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception:
            continue
    return pd.read_csv(path, low_memory=False)

def pick_col(cols, *keys):
    """열 이름 자동 감지: keys 중 하나가 포함된 첫 컬럼 반환"""
    low = {c.lower(): c for c in cols}
    for k in keys:
        for lc, orig in low.items():
            if k.lower() in lc:
                return orig
    return None

def to_float(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").astype(float)

def compute_P_mean_rowlevel(df: pd.DataFrame) -> pd.DataFrame:
    """44→8 감정: mean-매핑 (리뷰 행 단위) → P_{EMO}_mean 생성"""
    out = df.copy()
    cols_set = set(out.columns)
    for emo, candidates in GROUPS.items():
        base_cols = [c for c in candidates if c in cols_set]
        # 기본 라벨 합/개수
        if base_cols:
            base_vals = out[base_cols].fillna(0.0).to_numpy()
            base_sum  = base_vals.sum(axis=1)
            base_cnt  = base_vals.shape[1]
        else:
            base_sum = np.zeros(len(out))
            base_cnt = 0
        # 혼합 가중
        blend_sum, blend_wsum = np.zeros(len(out)), 0.0
        for col, wmap in BLEND.items():
            if (col in cols_set) and (emo in wmap):
                w = float(wmap[emo])
                blend_sum  += w * out[col].fillna(0.0).to_numpy()
                blend_wsum += w
        num = base_sum + blend_sum
        den = base_cnt + blend_wsum
        out[f"P_{emo}_mean"] = np.divide(num, den, out=np.zeros_like(num), where=(den!=0))
    return out

def compute_P_max_rowlevel(df: pd.DataFrame) -> pd.DataFrame:
    """44→8 감정: max-매핑 (리뷰 행 단위) → P_{EMO}_max 생성"""
    out = df.copy()
    cols_set = set(out.columns)
    for emo, candidates in GROUPS.items():
        blocks = []
        # 기본 후보: 원천 컬럼들의 행단위 max
        base = [c for c in candidates if c in cols_set]
        if base:
            blocks.append(out[base].fillna(0.0).to_numpy().max(axis=1))
        # 혼합 후보: (가중치×값)의 행단위 max
        blend_vecs = []
        for col, wmap in BLEND.items():
            if (col in cols_set) and (emo in wmap):
                w = float(wmap[emo])
                blend_vecs.append(w * out[col].fillna(0.0).to_numpy())
        if blend_vecs:
            blocks.append(np.column_stack(blend_vecs).max(axis=1))
        out[f"P_{emo}_max"] = np.column_stack(blocks).max(axis=1) if blocks else 0.0
    return out

def create_monthly_panel(df: pd.DataFrame, agg_map: dict) -> pd.DataFrame:
    """
    df: 리뷰 단위 행(이미 P_*_mean / P_*_max / polarity_pm1 포함)
    agg_map: {col_name: agg_func(str)}  # 예: {"P_JOY_mean": "mean", "polarity_pm1": "median"}
    """
    x = df.copy()
    x["month_start"] = x["DATE"].dt.to_period("M").dt.start_time
    has_text = ("감상평" in x.columns)

    # 그룹 집계
    specs = {c: (c, agg_map.get(c, "mean")) for c in agg_map.keys()}
    if has_text:
        specs["review_count"] = ("감상평", "count")
    for c in META_COLS:
        if c in x.columns:
            specs[c] = (c, "first")
    g = x.groupby(["영화명","month_start"]).agg(**specs).reset_index()

    if not has_text:
        sz = x.groupby(["영화명","month_start"]).size().reset_index(name="review_count")
        g = g.merge(sz, on=["영화명","month_start"], how="left")

    # 달력 스켈레톤
    if FIXED_RANGE_START and FIXED_RANGE_END:
        date_range = pd.date_range(start=pd.Timestamp(FIXED_RANGE_START),
                                   end=pd.Timestamp(FIXED_RANGE_END), freq="MS")
    else:
        date_range = pd.date_range(start=x["DATE"].dt.to_period("M").min().to_timestamp("MS"),
                                   end=x["DATE"].dt.to_period("M").max().to_timestamp("MS"), freq="MS")
    movies = x["영화명"].dropna().unique()
    skel = pd.MultiIndex.from_product([movies, date_range], names=["영화명","month_start"])
    panel = pd.DataFrame(index=skel).reset_index()
    panel = panel.merge(g, on=["영화명","month_start"], how="left")

    # 메타 전파
    for c in META_COLS:
        if c in panel.columns:
            panel[c] = panel.groupby("영화명")[c].transform("ffill").transform("bfill")

    # month_index (전역 0-base)
    first_m = panel["month_start"].min()
    panel["month_index"] = ((panel["month_start"].dt.year - first_m.year) * 12
                            + (panel["month_start"].dt.month - first_m.month))

    # gvar_month (OTT공개일 → month_index)
    if "OTT공개일" in panel.columns:
        panel["OTT공개일"] = pd.to_datetime(panel["OTT공개일"], errors="coerce")
        gstart = panel["OTT공개일"].dt.to_period("M").dt.start_time
        panel["gvar_month"] = np.where(
            pd.notna(gstart),
            (gstart.dt.year - first_m.year) * 12 + (gstart.dt.month - first_m.month),
            np.nan
        )
    else:
        panel["gvar_month"] = np.nan

    return panel

def full_months_diff(a: pd.Timestamp, b: pd.Timestamp):
    """만(Full) 개월 차이: ott.day < open.day이면 1개월 차감"""
    if pd.isna(a) or pd.isna(b): return np.nan
    ydiff = b.year - a.year
    mdiff = b.month - a.month
    total = ydiff*12 + mdiff
    if b.day < a.day:
        total -= 1
    return int(total)

def load_release_gap_months(path: str) -> pd.DataFrame:
    r = read_csv_kor(path)
    # 영화 이름 열 자동 인식
    mcol = pick_col(r.columns, "영화명","movie_name","title","moviename","movie")
    if mcol is None:
        raise KeyError("영화명(=movie name) 컬럼을 찾을 수 없습니다.")
    # 날짜 컬럼 자동 인식
    open_col = pick_col(r.columns, "open_date","개봉","release")
    ott_col  = pick_col(r.columns, "ott_date","ott","넷플릭스","netflix")
    if (open_col is None) or (ott_col is None):
        raise KeyError("open_date/ott_date 컬럼을 찾을 수 없습니다.")

    out = r[[mcol, open_col, ott_col]].rename(columns={mcol:"영화명", open_col:"open_date", ott_col:"ott_date"})
    out["open_date"] = pd.to_datetime(out["open_date"], errors="coerce")
    out["ott_date"]  = pd.to_datetime(out["ott_date"],  errors="coerce")
    out["ott_release_time_months"] = out.apply(lambda z: full_months_diff(z["open_date"], z["ott_date"]), axis=1)
    return out[["영화명","ott_release_time_months"]]

def coverage_summary(panel: pd.DataFrame, name: str):
    print(f"[{name}] rows={len(panel):,}, movies={panel['영화명'].nunique():,}, months={panel['month_start'].nunique():,}")
    if "review_count" in panel.columns:
        print("  review_count non-null:", int(panel["review_count"].notna().sum()))
    if "gvar_month" in panel.columns:
        print("  gvar_month non-null:", int(panel["gvar_month"].notna().sum()))

# =========================
# 3) MAIN
# =========================
def main():
    print("1) Load main CSV ...")
    df = read_csv_kor(INPUT_CSV)

    # 영화명 표준화 (영화명/movie_name 자동 감지)
    mcol = pick_col(df.columns, "영화명","movie_name")
    if mcol is None:
        raise KeyError("입력 CSV에서 영화명 컬럼을 찾을 수 없습니다.")
    if mcol != "영화명":
        df = df.rename(columns={mcol: "영화명"})

    print("2) Parse & typing ...")
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    df = df.dropna(subset=["DATE"]).copy()

    # 감정 원천 숫자형(0~1) 보정
    all_sources = set(sum(GROUPS.values(), [])) | set(BLEND.keys())
    for c in (all_sources & set(df.columns)):
        df[c] = pd.to_numeric(df[c], errors="coerce").clip(0, 1)

    # polarity [-1,1]
    if "polarity_pm1" not in df.columns:
        raise KeyError("입력 CSV에 'polarity_pm1'이 없습니다.")
    df["polarity_pm1"] = pd.to_numeric(df["polarity_pm1"], errors="coerce").clip(-1, 1)

    # OTT공개일 = 영화별 post_netflix==1 최초 DATE (있을 때)
    if "post_netflix" in df.columns:
        df["post_netflix"] = pd.to_numeric(df["post_netflix"], errors="coerce")
        first_ott = (
            df.loc[df["post_netflix"] == 1, ["영화명","DATE"]]
              .groupby("영화명", as_index=False)["DATE"].min()
              .rename(columns={"DATE":"OTT공개일"})
        )
        if "OTT공개일" in df.columns:
            df = df.drop(columns=["OTT공개일"])
        df = df.merge(first_ott, on="영화명", how="left")

    print("3) Build row-level P_*_mean (mean-mapping) ...")
    df_mean = compute_P_mean_rowlevel(df)

    print("4) Build row-level P_*_max (max-mapping) ...")
    df_both = compute_P_max_rowlevel(df_mean)  # df_mean에 이어 붙이기

    # === 월 집계 스펙 만들기 ===
    emo_mean_cols = [f"P_{e}_mean" for e in PRIMARY_EMOTIONS]
    emo_max_cols  = [f"P_{e}_max"  for e in PRIMARY_EMOTIONS]
    agg_map = {c: "mean" for c in (emo_mean_cols + emo_max_cols)}  # 같은 달이면 평균

    # polarity: median + mean
    agg_map["polarity_pm1"] = "median"  # 먼저 median으로 뽑아둔다(중간 패널에서 이름 바꿔 줄 것)

    print("5) Monthly panel for emotions + polarity_median ...")
    # 먼저 polarity는 median 패널을 만들고 이름 바꿔서 merge
    panel_med = create_monthly_panel(df_both, agg_map)
    panel_med = panel_med.rename(columns={"polarity_pm1":"polarity_median"})

    # polarity mean은 따로 한 번 더 만들어 병합
    panel_mean_only = create_monthly_panel(df_both, {"polarity_pm1":"mean"})
    panel_mean_only = panel_mean_only.rename(columns={"polarity_pm1":"polarity_mean"})

    # 병합
    merged = panel_med.merge(
        panel_mean_only[["영화명","month_start","polarity_mean"]],
        on=["영화명","month_start"], how="left"
    )

    # 6) 영화별 open~ott “만 개월” 차이 붙이기
    try:
        gap = load_release_gap_months(RELEASE_CSV)  # ["영화명","ott_release_time_months"]
        merged = merged.merge(gap, on="영화명", how="left")
    except Exception as e:
        print("⚠️ RELEASE_CSV 병합 스킵:", e)

    # 7) 컬럼 정렬/정리
    base_cols = [c for c in ["영화명","month_start","review_count","OTT공개일","대표국적","장르",
                             "month_index","gvar_month","ott_release_time_months"] if c in merged.columns]
    order = ["영화명","month_start"] + [c for c in base_cols if c not in ["영화명","month_start"]]
    order += emo_mean_cols + emo_max_cols + [c for c in ["polarity_median","polarity_mean"] if c in merged.columns]
    order += [c for c in merged.columns if c not in order]
    merged = merged[order]

    print("\n -> Save files ...")
    out_csv = "movie_monthly_panel_emotions_polarity.csv"
    out_dta = "movie_monthly_panel_emotions_polarity.dta"
    merged.to_csv(out_csv, index=False, encoding="utf-8-sig")
    merged.to_stata(out_dta, write_index=False, version=STATA_VERSION)

    coverage_summary(merged, "FINAL monthly panel")
    print(f"Saved:\n  - {out_csv}\n  - {out_dta}\n✨ Done.")

if __name__ == "__main__":
    main()


1) Load main CSV ...
2) Parse & typing ...
3) Build row-level P_*_mean (mean-mapping) ...
4) Build row-level P_*_max (max-mapping) ...
5) Monthly panel for emotions + polarity_median ...

 -> Save files ...
[FINAL monthly panel] rows=12,738, movies=193, months=66
  review_count non-null: 3481
  gvar_month non-null: 12738
Saved:
  - movie_monthly_panel_emotions_polarity.csv
  - movie_monthly_panel_emotions_polarity.dta
✨ Done.


# 별점까지

In [2]:
# -*- coding: utf-8 -*-
"""
Make ONE monthly panel that contains:
 - 8 primary emotions mapped from 44 labels in TWO ways at review-level:
     * mean-mapping  → P_{EMO}_mean
     * max-mapping   → P_{EMO}_max
   Then aggregated by movie×month using monthly AVERAGE (same-month reviews averaged)
 - polarity aggregated by movie×month as BOTH:
     * polarity_median (monthly median)
     * polarity_mean   (monthly mean)
 - ratings aggregated by movie×month as BOTH:
     * rating_median (monthly median)
     * rating_mean   (monthly mean)
 - meta columns, review_count, month_index, gvar_month (from first post_netflix==1 per movie)
 - movie-level months gap from open_date to ott_date: ott_release_time_months (constant per movie)

Output:
  - movie_monthly_panel_emotions_polarity.csv
  - movie_monthly_panel_emotions_polarity.dta
"""

import re
import pandas as pd
import numpy as np

# =========================
# 0) SETTINGS / PATH
# =========================
INPUT_CSV   = "movie_long_with_emotions and polarity_full.csv"  # 반드시 polarity_pm1 포함
RELEASE_CSV = "ott_release_by_movie(영화별 출시기간).csv"        # open/ott 날짜, 영화 키
STATA_VERSION = 118
META_COLS = ["OTT공개일", "대표국적", "장르"]

# (선택) 달력 범위 고정 (예: 2020-01~2025-12)
FIXED_RANGE_START = "2020-01-01"
FIXED_RANGE_END   = "2025-06-01"

# =========================
# 1) Emotion column mapping
# =========================
ALT = lambda *xs: list(xs)

JOY = ALT("기쁨_prob","행복_prob","즐거움/신남_prob","즐거움_신남_prob",
          "흐뭇함(귀여움/예쁨)_prob","흐뭇함_귀여움_예쁨__prob",
          "뿌듯함_prob","감동/감탄_prob","감동_감탄_prob","편안/쾌적_prob","편안_쾌적_prob")
TRUST = ALT("안심/신뢰_prob","안심_신뢰_prob","환영/호의_prob","환영_호의_prob",
            "존경_prob","아껴주는_prob")
FEAR  = ALT("공포/무서움_prob","공포_무서움_prob","불안/걱정_prob","불안_걱정_prob",
            "부담/안_내킴_prob","부담_안_내킴_prob")
SURPRISE = ALT("놀람_prob","경악_prob","당황/난처_prob","당황_난처_prob","깨달음_prob")
SADNESS  = ALT("슬픔_prob","절망_prob","서러움_prob","안타까움/실망_prob","안타까움_실망_prob",
               "재미없음_prob","힘듦/지침_prob","힘듦_지침_prob","패배/자기혐오_prob",
               "죄책감_prob","불쌍함/연민_prob","불쌍함_연민_prob","부끄러움_prob")
DISGUST  = ALT("역겨움/징그러움_prob","역겨움_징그러움_prob","한심함_prob","어이없음_prob",
               "지긋지긋_prob","증오/혐오_prob","증오_혐오_prob","우쭐댐/무시함_prob","우쭐댐_무시함_prob")
ANGER    = ALT("화남/분노_prob","화남_분노_prob","짜증_prob","불평/불만_prob","불평_불만_prob",
               "증오/혐오_prob","증오_혐오_prob","우쭐댐/무시함_prob","우쭐댐_무시함_prob")
ANTICIP  = ALT("기대감_prob","신기함/관심_prob","신기함_관심_prob","의심/불신_prob","의심_불신_prob")

PRIMARY_EMOTIONS = ["JOY","TRUST","FEAR","SURPRISE","SADNESS","DISGUST","ANGER","ANTICIP"]

GROUPS = {
    "JOY": JOY, "TRUST": TRUST, "FEAR": FEAR, "SURPRISE": SURPRISE,
    "SADNESS": SADNESS, "DISGUST": DISGUST, "ANGER": ANGER, "ANTICIP": ANTICIP
}

# 혼합(가중치 분배: mean-매핑에선 가중합/가중치, max-매핑에선 가중치 곱한 값으로 후보에 포함)
BLEND = {
    "증오/혐오_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "증오_혐오_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "우쭐댐/무시함_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "우쭐댐_무시함_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "고마움_prob": {"JOY": 0.5, "TRUST": 0.5},
    "부끄러움_prob": {"SADNESS": 0.5, "FEAR": 0.5},
}

# =========================
# 2) Utilities
# =========================
def read_csv_kor(path: str) -> pd.DataFrame:
    for enc in ["utf-8-sig","cp949","euc-kr","utf-8","latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception:
            continue
    return pd.read_csv(path, low_memory=False)

def pick_col(cols, *keys):
    """열 이름 자동 감지: keys 중 하나가 포함된 첫 컬럼 반환"""
    low = {c.lower(): c for c in cols}
    for k in keys:
        for lc, orig in low.items():
            if k.lower() in lc:
                return orig
    return None

def to_float(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").astype(float)

def parse_star_value(x):
    """
    문자열 별점에서 숫자만 추출: '★4.5', '4/5', '4.5점' 등 -> 4.5
    숫자면 그대로 반환. 실패하면 NaN.
    """
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x)
    m = re.search(r'(\d+(?:\.\d+)?)', s.replace(',', '.'))
    if m:
        try:
            return float(m.group(1))
        except:
            return np.nan
    return np.nan

def detect_rating_col(cols):
    """별점(평점) 컬럼 자동 감지"""
    candidates = ["별점", "평점", "rating", "stars", "score"]
    low = {c.lower(): c for c in cols}
    for key in candidates:
        for lc, orig in low.items():
            if key in lc:
                return orig
    return None

def compute_P_mean_rowlevel(df: pd.DataFrame) -> pd.DataFrame:
    """44→8 감정: mean-매핑 (리뷰 행 단위) → P_{EMO}_mean 생성"""
    out = df.copy()
    cols_set = set(out.columns)
    for emo, candidates in GROUPS.items():
        base_cols = [c for c in candidates if c in cols_set]
        # 기본 라벨 합/개수
        if base_cols:
            base_vals = out[base_cols].fillna(0.0).to_numpy()
            base_sum  = base_vals.sum(axis=1)
            base_cnt  = base_vals.shape[1]
        else:
            base_sum = np.zeros(len(out))
            base_cnt = 0
        # 혼합 가중
        blend_sum, blend_wsum = np.zeros(len(out)), 0.0
        for col, wmap in BLEND.items():
            if (col in cols_set) and (emo in wmap):
                w = float(wmap[emo])
                blend_sum  += w * out[col].fillna(0.0).to_numpy()
                blend_wsum += w
        num = base_sum + blend_sum
        den = base_cnt + blend_wsum
        out[f"P_{emo}_mean"] = np.divide(num, den, out=np.zeros_like(num), where=(den!=0))
    return out

def compute_P_max_rowlevel(df: pd.DataFrame) -> pd.DataFrame:
    """44→8 감정: max-매핑 (리뷰 행 단위) → P_{EMO}_max 생성"""
    out = df.copy()
    cols_set = set(out.columns)
    for emo, candidates in GROUPS.items():
        blocks = []
        # 기본 후보: 원천 컬럼들의 행단위 max
        base = [c for c in candidates if c in cols_set]
        if base:
            blocks.append(out[base].fillna(0.0).to_numpy().max(axis=1))
        # 혼합 후보: (가중치×값)의 행단위 max
        blend_vecs = []
        for col, wmap in BLEND.items():
            if (col in cols_set) and (emo in wmap):
                w = float(wmap[emo])
                blend_vecs.append(w * out[col].fillna(0.0).to_numpy())
        if blend_vecs:
            blocks.append(np.column_stack(blend_vecs).max(axis=1))
        out[f"P_{emo}_max"] = np.column_stack(blocks).max(axis=1) if blocks else 0.0
    return out

def create_monthly_panel(df: pd.DataFrame, agg_map: dict) -> pd.DataFrame:
    """
    df: 리뷰 단위 행(이미 P_*_mean / P_*_max / polarity_pm1 / rating_value 포함 가능)
    agg_map: {col_name: agg_func(str)}  # 예: {"P_JOY_mean": "mean", "polarity_pm1": "median"}
    """
    x = df.copy()
    x["month_start"] = x["DATE"].dt.to_period("M").dt.start_time
    has_text = ("감상평" in x.columns)

    # 그룹 집계
    specs = {c: (c, agg_map.get(c, "mean")) for c in agg_map.keys()}
    if has_text:
        specs["review_count"] = ("감상평", "count")
    for c in META_COLS:
        if c in x.columns:
            specs[c] = (c, "first")
    g = x.groupby(["영화명","month_start"]).agg(**specs).reset_index()

    if not has_text:
        sz = x.groupby(["영화명","month_start"]).size().reset_index(name="review_count")
        g = g.merge(sz, on=["영화명","month_start"], how="left")

    # 달력 스켈레톤
    if FIXED_RANGE_START and FIXED_RANGE_END:
        date_range = pd.date_range(start=pd.Timestamp(FIXED_RANGE_START),
                                   end=pd.Timestamp(FIXED_RANGE_END), freq="MS")
    else:
        date_range = pd.date_range(start=x["DATE"].dt.to_period("M").min().to_timestamp("MS"),
                                   end=x["DATE"].dt.to_period("M").max().to_timestamp("MS"), freq="MS")
    movies = x["영화명"].dropna().unique()
    skel = pd.MultiIndex.from_product([movies, date_range], names=["영화명","month_start"])
    panel = pd.DataFrame(index=skel).reset_index()
    panel = panel.merge(g, on=["영화명","month_start"], how="left")

    # 메타 전파
    for c in META_COLS:
        if c in panel.columns:
            panel[c] = panel.groupby("영화명")[c].transform("ffill").transform("bfill")

    # month_index (전역 0-base)
    first_m = panel["month_start"].min()
    panel["month_index"] = ((panel["month_start"].dt.year - first_m.year) * 12
                            + (panel["month_start"].dt.month - first_m.month))

    # gvar_month (OTT공개일 → month_index)
    if "OTT공개일" in panel.columns:
        panel["OTT공개일"] = pd.to_datetime(panel["OTT공개일"], errors="coerce")
        gstart = panel["OTT공개일"].dt.to_period("M").dt.start_time
        panel["gvar_month"] = np.where(
            pd.notna(gstart),
            (gstart.dt.year - first_m.year) * 12 + (gstart.dt.month - first_m.month),
            np.nan
        )
    else:
        panel["gvar_month"] = np.nan

    return panel

def full_months_diff(a: pd.Timestamp, b: pd.Timestamp):
    """만(Full) 개월 차이: ott.day < open.day이면 1개월 차감"""
    if pd.isna(a) or pd.isna(b): return np.nan
    ydiff = b.year - a.year
    mdiff = b.month - a.month
    total = ydiff*12 + mdiff
    if b.day < a.day:
        total -= 1
    return int(total)

def load_release_gap_months(path: str) -> pd.DataFrame:
    r = read_csv_kor(path)
    # 영화 이름 열 자동 인식
    mcol = pick_col(r.columns, "영화명","movie_name","title","moviename","movie")
    if mcol is None:
        raise KeyError("영화명(=movie name) 컬럼을 찾을 수 없습니다.")
    # 날짜 컬럼 자동 인식
    open_col = pick_col(r.columns, "open_date","개봉","release")
    ott_col  = pick_col(r.columns, "ott_date","ott","넷플릭스","netflix")
    if (open_col is None) or (ott_col is None):
        raise KeyError("open_date/ott_date 컬럼을 찾을 수 없습니다.")

    out = r[[mcol, open_col, ott_col]].rename(columns={mcol:"영화명", open_col:"open_date", ott_col:"ott_date"})
    out["open_date"] = pd.to_datetime(out["open_date"], errors="coerce")
    out["ott_date"]  = pd.to_datetime(out["ott_date"],  errors="coerce")
    out["ott_release_time_months"] = out.apply(lambda z: full_months_diff(z["open_date"], z["ott_date"]), axis=1)
    return out[["영화명","ott_release_time_months"]]

def coverage_summary(panel: pd.DataFrame, name: str):
    print(f"[{name}] rows={len(panel):,}, movies={panel['영화명'].nunique():,}, months={panel['month_start'].nunique():,}")
    if "review_count" in panel.columns:
        print("  review_count non-null:", int(panel["review_count"].notna().sum()))
    if "gvar_month" in panel.columns:
        print("  gvar_month non-null:", int(panel["gvar_month"].notna().sum()))

# =========================
# 3) MAIN
# =========================
def main():
    print("1) Load main CSV ...")
    df = read_csv_kor(INPUT_CSV)

    # 영화명 표준화 (영화명/movie_name 자동 감지)
    mcol = pick_col(df.columns, "영화명","movie_name")
    if mcol is None:
        raise KeyError("입력 CSV에서 영화명 컬럼을 찾을 수 없습니다.")
    if mcol != "영화명":
        df = df.rename(columns={mcol: "영화명"})

    print("2) Parse & typing ...")
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    df = df.dropna(subset=["DATE"]).copy()

    # 감정 원천 숫자형(0~1) 보정
    all_sources = set(sum(GROUPS.values(), [])) | set(BLEND.keys())
    for c in (all_sources & set(df.columns)):
        df[c] = pd.to_numeric(df[c], errors="coerce").clip(0, 1)

    # polarity [-1,1]
    if "polarity_pm1" not in df.columns:
        raise KeyError("입력 CSV에 'polarity_pm1'이 없습니다.")
    df["polarity_pm1"] = pd.to_numeric(df["polarity_pm1"], errors="coerce").clip(-1, 1)

    # --- 별점(평점): 자동 감지 + 숫자화 ---
    rating_col = detect_rating_col(df.columns)
    if rating_col is not None:
        df["__rating_raw__"] = df[rating_col]
        df["rating_value"] = df["__rating_raw__"].map(parse_star_value).astype(float)
        has_rating = True
    else:
        has_rating = False
        print("⚠️ 별점/평점 컬럼을 찾지 못해 rating 집계를 생략합니다.")

    # OTT공개일 = 영화별 post_netflix==1 최초 DATE (있을 때)
    if "post_netflix" in df.columns:
        df["post_netflix"] = pd.to_numeric(df["post_netflix"], errors="coerce")
        first_ott = (
            df.loc[df["post_netflix"] == 1, ["영화명","DATE"]]
              .groupby("영화명", as_index=False)["DATE"].min()
              .rename(columns={"DATE":"OTT공개일"})
        )
        if "OTT공개일" in df.columns:
            df = df.drop(columns=["OTT공개일"])
        df = df.merge(first_ott, on="영화명", how="left")

    print("3) Build row-level P_*_mean (mean-mapping) ...")
    df_mean = compute_P_mean_rowlevel(df)

    print("4) Build row-level P_*_max (max-mapping) ...")
    df_both = compute_P_max_rowlevel(df_mean)  # df_mean에 이어 붙이기

    # === 월 집계 스펙 만들기 ===
    emo_mean_cols = [f"P_{e}_mean" for e in PRIMARY_EMOTIONS]
    emo_max_cols  = [f"P_{e}_max"  for e in PRIMARY_EMOTIONS]
    agg_map = {c: "mean" for c in (emo_mean_cols + emo_max_cols)}  # 같은 달이면 평균

    # polarity: median + mean
    agg_map["polarity_pm1"] = "median"  # 먼저 median 패널

    print("5) Monthly panel for emotions + polarity_median ...")
    panel_med = create_monthly_panel(df_both, agg_map).rename(columns={"polarity_pm1":"polarity_median"})

    # polarity mean은 따로 한 번 더
    panel_mean_only = create_monthly_panel(df_both, {"polarity_pm1":"mean"}) \
                        .rename(columns={"polarity_pm1":"polarity_mean"})

    # 병합 시작
    merged = panel_med.merge(
        panel_mean_only[["영화명","month_start","polarity_mean"]],
        on=["영화명","month_start"], how="left"
    )

    # ---- rating (별점): monthly mean & median ----
    if has_rating:
        panel_rating_mean = create_monthly_panel(df_both, {"rating_value":"mean"}) \
                                .rename(columns={"rating_value":"rating_mean"})
        panel_rating_median = create_monthly_panel(df_both, {"rating_value":"median"}) \
                                .rename(columns={"rating_value":"rating_median"})
        merged = merged.merge(
            panel_rating_mean[["영화명","month_start","rating_mean"]],
            on=["영화명","month_start"], how="left"
        ).merge(
            panel_rating_median[["영화명","month_start","rating_median"]],
            on=["영화명","month_start"], how="left"
        )

    # 6) 영화별 open~ott “만 개월” 차이 붙이기
    try:
        gap = load_release_gap_months(RELEASE_CSV)  # ["영화명","ott_release_time_months"]
        merged = merged.merge(gap, on="영화명", how="left")
    except Exception as e:
        print("⚠️ RELEASE_CSV 병합 스킵:", e)

    # 7) 컬럼 정렬/정리
    base_cols = [c for c in ["영화명","month_start","review_count","OTT공개일","대표국적","장르",
                             "month_index","gvar_month","ott_release_time_months"] if c in merged.columns]
    order = ["영화명","month_start"] + [c for c in base_cols if c not in ["영화명","month_start"]]
    order += emo_mean_cols + emo_max_cols
    if "polarity_median" in merged.columns: order += ["polarity_median"]
    if "polarity_mean"   in merged.columns: order += ["polarity_mean"]
    if "rating_median"   in merged.columns: order += ["rating_median"]
    if "rating_mean"     in merged.columns: order += ["rating_mean"]
    order += [c for c in merged.columns if c not in order]
    merged = merged[order]

    print("\n -> Save files ...")
    out_csv = "movie_monthly_panel_emotions_polarity.csv"
    out_dta = "movie_monthly_panel_emotions_polarity.dta"
    merged.to_csv(out_csv, index=False, encoding="utf-8-sig")
    merged.to_stata(out_dta, write_index=False, version=STATA_VERSION)

    coverage_summary(merged, "FINAL monthly panel")
    print(f"Saved:\n  - {out_csv}\n  - {out_dta}\n✨ Done.")

if __name__ == "__main__":
    main()


1) Load main CSV ...
2) Parse & typing ...
3) Build row-level P_*_mean (mean-mapping) ...
4) Build row-level P_*_max (max-mapping) ...
5) Monthly panel for emotions + polarity_median ...

 -> Save files ...
[FINAL monthly panel] rows=12,738, movies=193, months=66
  review_count non-null: 3481
  gvar_month non-null: 12738
Saved:
  - movie_monthly_panel_emotions_polarity.csv
  - movie_monthly_panel_emotions_polarity.dta
✨ Done.


# max 이상해서 다시

In [5]:
# -*- coding: utf-8 -*-
"""
Make ONE monthly panel that contains (LEGACY style for MAX):
 - 8 primary emotions mapped from 44 labels at review-level in TWO ways:
     * mean-mapping  → P_{EMO}_mean  (BLEND weighted: e.g., hate→0.5/0.5)
     * max-mapping   → P_{EMO}_max   (LEGACY: BLEND cols included AS-IS, no weights)
   Then aggregated by movie×month using monthly AVERAGE (same-month reviews averaged)
 - polarity aggregated by movie×month as BOTH:
     * polarity_median (monthly median)
     * polarity_mean   (monthly mean)
 - ratings aggregated by movie×month as BOTH (if a rating-like column exists):
     * rating_median (monthly median)
     * rating_mean   (monthly mean)
 - meta columns, review_count, month_index, gvar_month (from first post_netflix==1 per movie)
 - movie-level months gap from open_date to ott_date: ott_release_time_months (constant per movie)

Outputs:
  - movie_monthly_panel_emotions_polarity.csv
  - movie_monthly_panel_emotions_polarity.dta
"""

import re
import pandas as pd
import numpy as np

# =========================
# 0) SETTINGS / PATH
# =========================
INPUT_CSV   = "movie_long_with_emotions and polarity_full.csv"  # 반드시 polarity_pm1 포함
RELEASE_CSV = "ott_release_by_movie(영화별 출시기간).csv"        # open/ott 날짜, 영화 키 (선택)
STATA_VERSION = 118
META_COLS = ["OTT공개일", "대표국적", "장르"]

# (선택) 달력 범위 고정 (예: 2020-01~2025-12)
FIXED_RANGE_START ="2020-01-01"  # 예: "2020-01-01"
FIXED_RANGE_END   ="2025-12-01"  # 예: "2025-12-01"

# =========================
# 1) Emotion column mapping
# =========================
ALT = lambda *xs: list(xs)

JOY = ALT("기쁨_prob","행복_prob","즐거움/신남_prob","즐거움_신남_prob",
          "흐뭇함(귀여움/예쁨)_prob","흐뭇함_귀여움_예쁨__prob",
          "뿌듯함_prob","감동/감탄_prob","감동_감탄_prob","편안/쾌적_prob","편안_쾌적_prob")
TRUST = ALT("안심/신뢰_prob","안심_신뢰_prob","환영/호의_prob","환영_호의_prob",
            "존경_prob","아껴주는_prob")
FEAR  = ALT("공포/무서움_prob","공포_무서움_prob","불안/걱정_prob","불안_걱정_prob",
            "부담/안_내킴_prob","부담_안_내킴_prob")
SURPRISE = ALT("놀람_prob","경악_prob","당황/난처_prob","당황_난처_prob","깨달음_prob")
SADNESS  = ALT("슬픔_prob","절망_prob","서러움_prob","안타까움/실망_prob","안타까움_실망_prob",
               "재미없음_prob","힘듦/지침_prob","힘듦_지침_prob","패배/자기혐오_prob",
               "죄책감_prob","불쌍함/연민_prob","불쌍함_연민_prob","부끄러움_prob")
DISGUST  = ALT("역겨움/징그러움_prob","역겨움_징그러움_prob","한심함_prob","어이없음_prob",
               "지긋지긋_prob","증오/혐오_prob","증오_혐오_prob","우쭐댐/무시함_prob","우쭐댐_무시함_prob")
ANGER    = ALT("화남/분노_prob","화남_분노_prob","짜증_prob","불평/불만_prob","불평_불만_prob",
               "증오/혐오_prob","증오_혐오_prob","우쭐댐/무시함_prob","우쭐댐_무시함_prob")
ANTICIP  = ALT("기대감_prob","신기함/관심_prob","신기함_관심_prob","의심/불신_prob","의심_불신_prob")

PRIMARY_EMOTIONS = ["JOY","TRUST","FEAR","SURPRISE","SADNESS","DISGUST","ANGER","ANTICIP"]

GROUPS = {
    "JOY": JOY, "TRUST": TRUST, "FEAR": FEAR, "SURPRISE": SURPRISE,
    "SADNESS": SADNESS, "DISGUST": DISGUST, "ANGER": ANGER, "ANTICIP": ANTICIP
}

# 혼합(가중치 분배: mean-매핑에선 가중합/가중치, MAX(LEGACY)에서는 '그냥 후보열'로 포함)
BLEND = {
    "증오/혐오_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "증오_혐오_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "우쭐댐/무시함_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "우쭐댐_무시함_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "고마움_prob": {"JOY": 0.5, "TRUST": 0.5},
    "부끄러움_prob": {"SADNESS": 0.5, "FEAR": 0.5},
}

# =========================
# 2) Utilities
# =========================
def read_csv_kor(path: str) -> pd.DataFrame:
    for enc in ["utf-8-sig","cp949","euc-kr","utf-8","latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception:
            continue
    return pd.read_csv(path, low_memory=False)

def pick_col(cols, *keys):
    """열 이름 자동 감지: keys 중 하나가 포함된 첫 컬럼 반환"""
    low = {c.lower(): c for c in cols}
    for k in keys:
        for lc, orig in low.items():
            if k.lower() in lc:
                return orig
    return None

def to_float(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").astype(float)

def parse_star_value(x):
    """문자열 별점에서 숫자만 추출: '★4.5', '4/5', '4.5점' 등 -> 4.5"""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x)
    m = re.search(r'(\d+(?:\.\d+)?)', s.replace(',', '.'))
    if m:
        try:
            return float(m.group(1))
        except:
            return np.nan
    return np.nan

def detect_rating_col(cols):
    """별점(평점) 컬럼 자동 감지"""
    candidates = ["별점", "평점", "rating", "stars", "score"]
    low = {c.lower(): c for c in cols}
    for key in candidates:
        for lc, orig in low.items():
            if key in lc:
                return orig
    return None

# --- Review-level mapping (mean / legacy max) ---
def compute_P_mean_rowlevel(df: pd.DataFrame) -> pd.DataFrame:
    """44→8 감정: mean-매핑 (리뷰 행 단위) → P_{EMO}_mean 생성 (BLEND 가중)"""
    out = df.copy()
    cols_set = set(out.columns)
    for emo, candidates in GROUPS.items():
        base_cols = [c for c in candidates if c in cols_set]
        # 기본 라벨 합/개수
        if base_cols:
            base_vals = out[base_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy()
            base_sum  = base_vals.sum(axis=1)
            base_cnt  = base_vals.shape[1]
        else:
            base_sum = np.zeros(len(out))
            base_cnt = 0
        # 혼합 가중(가중합/가중치)
        blend_sum, blend_wsum = np.zeros(len(out)), 0.0
        for col, wmap in BLEND.items():
            if (col in cols_set) and (emo in wmap):
                w = float(wmap[emo])
                blend_sum  += w * out[col].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy()
                blend_wsum += w
        num = base_sum + blend_sum
        den = base_cnt + blend_wsum
        out[f"P_{emo}_mean"] = np.divide(num, den, out=np.zeros_like(num), where=(den!=0))
    return out

def compute_P_max_rowlevel_legacy(df: pd.DataFrame) -> pd.DataFrame:
    """
    44→8 감정: LEGACY max-매핑 (리뷰 행 단위) → P_{EMO}_max 생성
    - BLEND 컬럼을 가중치 곱하지 않고 '그냥 후보열'로 포함
    - 결측을 0으로 강제 치환하지 않음 (기본 skipna=True에 의존)
    """
    out = df.copy()
    cols_set = set(out.columns)
    for emo, candidates in GROUPS.items():
        # 기본 후보
        base = [c for c in candidates if c in cols_set]
        mats = []
        if base:
            mats.append(out[base].apply(pd.to_numeric, errors="coerce").max(axis=1))  # skipna=True

        # BLEND 후보(무가중, 그냥 포함)
        blend_exist = [c for c in BLEND.keys() if c in cols_set]
        if blend_exist:
            mats.append(out[blend_exist].apply(pd.to_numeric, errors="coerce").max(axis=1))

        # 최종 MAX
        if mats:
            out[f"P_{emo}_max"] = pd.concat(mats, axis=1).max(axis=1)
        else:
            out[f"P_{emo}_max"] = np.nan
    return out

# --- Monthly panel builder ---
def create_monthly_panel(df: pd.DataFrame, agg_map: dict) -> pd.DataFrame:
    """
    df: 리뷰 단위 행(이미 P_*_mean / P_*_max / polarity_pm1 / rating_value 포함 가능)
    agg_map: {col_name: agg_func(str)}  # 예: {"P_JOY_mean": "mean", "polarity_pm1": "median"}
    """
    x = df.copy()
    x["month_start"] = x["DATE"].dt.to_period("M").dt.start_time
    has_text = ("감상평" in x.columns)

    # 그룹 집계
    specs = {c: (c, agg_map.get(c, "mean")) for c in agg_map.keys()}
    if has_text:
        specs["review_count"] = ("감상평", "count")
    for c in META_COLS:
        if c in x.columns:
            specs[c] = (c, "first")
    g = x.groupby(["영화명","month_start"]).agg(**specs).reset_index()

    if not has_text:
        sz = x.groupby(["영화명","month_start"]).size().reset_index(name="review_count")
        g = g.merge(sz, on=["영화명","month_start"], how="left")

    # 달력 스켈레톤
    if FIXED_RANGE_START and FIXED_RANGE_END:
        date_range = pd.date_range(start=pd.Timestamp(FIXED_RANGE_START),
                                   end=pd.Timestamp(FIXED_RANGE_END), freq="MS")
    else:
        date_range = pd.date_range(start=x["DATE"].dt.to_period("M").min().to_timestamp("MS"),
                                   end=x["DATE"].dt.to_period("M").max().to_timestamp("MS"), freq="MS")
    movies = x["영화명"].dropna().unique()
    skel = pd.MultiIndex.from_product([movies, date_range], names=["영화명","month_start"])
    panel = pd.DataFrame(index=skel).reset_index()
    panel = panel.merge(g, on=["영화명","month_start"], how="left")

    # 메타 전파
    for c in META_COLS:
        if c in panel.columns:
            panel[c] = panel.groupby("영화명")[c].transform("ffill").transform("bfill")

    # month_index (전역 0-base)
    first_m = panel["month_start"].min()
    panel["month_index"] = ((panel["month_start"].dt.year - first_m.year) * 12
                            + (panel["month_start"].dt.month - first_m.month))

    # gvar_month (OTT공개일 → month_index)
    if "OTT공개일" in panel.columns:
        panel["OTT공개일"] = pd.to_datetime(panel["OTT공개일"], errors="coerce")
        gstart = panel["OTT공개일"].dt.to_period("M").dt.start_time
        panel["gvar_month"] = np.where(
            pd.notna(gstart),
            (gstart.dt.year - first_m.year) * 12 + (gstart.dt.month - first_m.month),
            np.nan
        )
    else:
        panel["gvar_month"] = np.nan

    return panel

def full_months_diff(a: pd.Timestamp, b: pd.Timestamp):
    """만(Full) 개월 차이: ott.day < open.day이면 1개월 차감"""
    if pd.isna(a) or pd.isna(b): return np.nan
    ydiff = b.year - a.year
    mdiff = b.month - a.month
    total = ydiff*12 + mdiff
    if b.day < a.day:
        total -= 1
    return int(total)

def load_release_gap_months(path: str) -> pd.DataFrame:
    r = read_csv_kor(path)
    # 영화 이름 열 자동 인식
    mcol = pick_col(r.columns, "영화명","movie_name","title","moviename","movie")
    if mcol is None:
        raise KeyError("영화명(=movie name) 컬럼을 찾을 수 없습니다.")
    # 날짜 컬럼 자동 인식
    open_col = pick_col(r.columns, "open_date","개봉","release")
    ott_col  = pick_col(r.columns, "ott_date","ott","넷플릭스","netflix")
    if (open_col is None) or (ott_col is None):
        raise KeyError("open_date/ott_date 컬럼을 찾을 수 없습니다.")

    out = r[[mcol, open_col, ott_col]].rename(columns={mcol:"영화명", open_col:"open_date", ott_col:"ott_date"})
    out["open_date"] = pd.to_datetime(out["open_date"], errors="coerce")
    out["ott_date"]  = pd.to_datetime(out["ott_date"],  errors="coerce")
    out["ott_release_time_months"] = out.apply(lambda z: full_months_diff(z["open_date"], z["ott_date"]), axis=1)
    return out[["영화명","ott_release_time_months"]]

def coverage_summary(panel: pd.DataFrame, name: str):
    print(f"[{name}] rows={len(panel):,}, movies={panel['영화명'].nunique():,}, months={panel['month_start'].nunique():,}")
    if "review_count" in panel.columns:
        print("  review_count non-null:", int(panel["review_count"].notna().sum()))
    if "gvar_month" in panel.columns:
        print("  gvar_month non-null:", int(panel["gvar_month"].notna().sum()))

# =========================
# 3) MAIN
# =========================
def main():
    print("1) Load main CSV ...")
    df = read_csv_kor(INPUT_CSV)

    # 영화명 표준화 (영화명/movie_name 자동 감지)
    mcol = pick_col(df.columns, "영화명","movie_name")
    if mcol is None:
        raise KeyError("입력 CSV에서 영화명 컬럼을 찾을 수 없습니다.")
    if mcol != "영화명":
        df = df.rename(columns={mcol: "영화명"})

    print("2) Parse & typing ...")
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    df = df.dropna(subset=["DATE"]).copy()

    # 감정 원천 숫자형(0~1) 보정
    all_sources = set(sum(GROUPS.values(), [])) | set(BLEND.keys())
    for c in (all_sources & set(df.columns)):
        df[c] = pd.to_numeric(df[c], errors="coerce").clip(0, 1)

    # polarity [-1,1]
    if "polarity_pm1" not in df.columns:
        raise KeyError("입력 CSV에 'polarity_pm1'이 없습니다.")
    df["polarity_pm1"] = pd.to_numeric(df["polarity_pm1"], errors="coerce").clip(-1, 1)

    # --- 별점(평점): 자동 감지 + 숫자화 ---
    rating_col = detect_rating_col(df.columns)
    if rating_col is not None:
        df["__rating_raw__"] = df[rating_col]
        df["rating_value"] = df["__rating_raw__"].map(parse_star_value).astype(float)
        has_rating = True
    else:
        has_rating = False
        print("⚠️ 별점/평점 컬럼을 찾지 못해 rating 집계를 생략합니다.")

    # OTT공개일 = 영화별 post_netflix==1 최초 DATE (있을 때)
    if "post_netflix" in df.columns:
        df["post_netflix"] = pd.to_numeric(df["post_netflix"], errors="coerce")
        first_ott = (
            df.loc[df["post_netflix"] == 1, ["영화명","DATE"]]
              .groupby("영화명", as_index=False)["DATE"].min()
              .rename(columns={"DATE":"OTT공개일"})
        )
        if "OTT공개일" in df.columns:
            df = df.drop(columns=["OTT공개일"])
        df = df.merge(first_ott, on="영화명", how="left")

    # --- Review-level 44→8 mapping ---
    print("3) Build row-level P_*_mean (mean-mapping, with BLEND weights) ...")
    df_mean = compute_P_mean_rowlevel(df)

    print("4) Build row-level P_*_max (LEGACY max-mapping, BLEND as plain candidates) ...")
    df_both = compute_P_max_rowlevel_legacy(df_mean)  # df_mean에 이어 붙이기

    # === 월 집계 스펙 ===
    emo_mean_cols = [f"P_{e}_mean" for e in PRIMARY_EMOTIONS]
    emo_max_cols  = [f"P_{e}_max"  for e in PRIMARY_EMOTIONS]
    agg_map = {c: "mean" for c in (emo_mean_cols + emo_max_cols)}  # 같은 달이면 평균

    # polarity: median + mean
    agg_map["polarity_pm1"] = "median"  # 먼저 median으로 만들 패널

    print("5) Monthly panel for emotions + polarity_median ...")
    panel_med = create_monthly_panel(df_both, agg_map).rename(columns={"polarity_pm1":"polarity_median"})

    # polarity mean은 따로 한 번 더
    panel_mean_only = create_monthly_panel(df_both, {"polarity_pm1":"mean"}) \
                        .rename(columns={"polarity_pm1":"polarity_mean"})

    # 병합 시작
    merged = panel_med.merge(
        panel_mean_only[["영화명","month_start","polarity_mean"]],
        on=["영화명","month_start"], how="left"
    )

    # ---- rating (별점): monthly mean & median ----
    if has_rating:
        panel_rating_mean = create_monthly_panel(df_both, {"rating_value":"mean"}) \
                                .rename(columns={"rating_value":"rating_mean"})
        panel_rating_median = create_monthly_panel(df_both, {"rating_value":"median"}) \
                                .rename(columns={"rating_value":"rating_median"})
        merged = merged.merge(
            panel_rating_mean[["영화명","month_start","rating_mean"]],
            on=["영화명","month_start"], how="left"
        ).merge(
            panel_rating_median[["영화명","month_start","rating_median"]],
            on=["영화명","month_start"], how="left"
        )

    # 6) 영화별 open~ott “만 개월” 차이 붙이기 (선택)
    try:
        gap = load_release_gap_months(RELEASE_CSV)  # ["영화명","ott_release_time_months"]
        merged = merged.merge(gap, on="영화명", how="left")
    except Exception as e:
        print("⚠️ RELEASE_CSV 병합 스킵:", e)

    # 7) 컬럼 정렬/정리
    base_cols = [c for c in ["영화명","month_start","review_count","OTT공개일","대표국적","장르",
                             "month_index","gvar_month","ott_release_time_months"] if c in merged.columns]
    order = ["영화명","month_start"] + [c for c in base_cols if c not in ["영화명","month_start"]]
    order += emo_mean_cols + emo_max_cols
    if "polarity_median" in merged.columns: order += ["polarity_median"]
    if "polarity_mean"   in merged.columns: order += ["polarity_mean"]
    if "rating_median"   in merged.columns: order += ["rating_median"]
    if "rating_mean"     in merged.columns: order += ["rating_mean"]
    order += [c for c in merged.columns if c not in order]
    merged = merged[order]

    print("\n -> Save files ...")
    out_csv = "movie_monthly_panel_emotions_polarity.csv"
    out_dta = "movie_monthly_panel_emotions_polarity.dta"
    merged.to_csv(out_csv, index=False, encoding="utf-8-sig")
    merged.to_stata(out_dta, write_index=False, version=STATA_VERSION)

    coverage_summary(merged, "FINAL monthly panel")
    print(f"Saved:\n  - {out_csv}\n  - {out_dta}\n✨ Done.")

if __name__ == "__main__":
    main()


1) Load main CSV ...
2) Parse & typing ...
3) Build row-level P_*_mean (mean-mapping, with BLEND weights) ...
4) Build row-level P_*_max (LEGACY max-mapping, BLEND as plain candidates) ...
5) Monthly panel for emotions + polarity_median ...

 -> Save files ...
[FINAL monthly panel] rows=13,896, movies=193, months=72
  review_count non-null: 3481
  gvar_month non-null: 13896
Saved:
  - movie_monthly_panel_emotions_polarity.csv
  - movie_monthly_panel_emotions_polarity.dta
✨ Done.
